# A3 · Fine-tune Conv-TasNet on Hindi mixtures (Google Colab)

This notebook carries out improvement **A3**: keep training the released Conv-TasNet (`sepnoisy`: English read speech,
WHAM! noise) on Hindi two-speaker mixtures with our own noise, then check whether it separates our overlaps better
and whether that lowers the transcript error.

**Rules (written before running, so the result can't be chosen after seeing it):**
1. **Training data:** IndicVoices Hindi *train* shard(s), the pool the dev set was drawn from, **minus every speaker
   in the dev and test conversations and every speaker of the valid split** (the test pool). Noise: only the **dev
   halves** of the village and market soundscapes. 3-second segments.
2. **Checkpoint choice:** the training step with the best SI-SDR improvement on the 128 **dev** overlap regions.
3. **Keep it only if it wins on dev**, meaning all three of:
   - dev overlap SI-SDR improvement is higher than the released model's,
   - dev cpWER with the true timeline (splice-0.5) is lower,
   - dev cpWER with our diarization (B-spectral, splice-0.5) is not higher.
4. **The dev decision is written to Drive before the test set is touched.** The test set is then run once, so that
   every experiment is complete, but the fine-tuned model counts as an improvement only if it won on dev; the report
   says which.

**Before you start:**
- Upload **`whospoke_colab_bundle.zip`** (~350 MB, made by `scripts/make_colab_bundle.py`) to the top of your Google
  Drive (`MyDrive/`). You do **not** need the whole repo or its 6.5 GB data folder: the bundle has the code, the dev
  and test conversations and the dev noise; IndicVoices is downloaded here on Colab.
- Runtime → Change runtime type → **T4 GPU**.
- Add your Hugging Face token under **Secrets** (🔑 icon on the left) with the name `HF_TOKEN`, and turn on
  "Notebook access". The account must have accepted the terms of `pyannote/segmentation-3.0`,
  `ai4bharat/indic-conformer-600m-multilingual` and the `ai4bharat/IndicVoices` **dataset** (cell 0 checks all three).

**Time on a T4:** about 2.5–3 hours in all (training ~45 min; dev checks ~30 min; test ~60 min). The work runs on
Colab's fast local disk; every result and checkpoint is written straight to `MyDrive/whospoke_colab_out/`. If Colab
disconnects, run all cells again: finished work is skipped and training resumes from its last checkpoint.

**What to bring back:** the folder `MyDrive/whospoke_colab_out/` (download it as a zip): `results/` (with
`finetune_convtasnet/REPORT.md`) and `models/convtasnet_hindi.pth` (20 MB).

In [ ]:
# ---- settings -------------------------------------------------------------------------------
BUNDLE = "/content/drive/MyDrive/whospoke_colab_bundle.zip"   # the uploaded bundle
OUT = "/content/drive/MyDrive/whospoke_colab_out"             # results and the model are kept here
EXTRA_SHARDS = 1      # extra IndicVoices train shards to download (~500 MB each); 0 = only the local shard 0
STEPS = 8000          # training steps (batch 8, 3 s each); stops earlier if dev stops improving
BATCH = 8

## 0 · Setup: Drive, packages, Hugging Face

In [ ]:
import os, sys, subprocess, json, shutil, zipfile
from pathlib import Path
from google.colab import drive, userdata
drive.mount("/content/drive")
REPO = Path("/content/whospoke")
if not (REPO / "src" / "whospoke").exists():               # unpack once per Colab session, on the fast local disk
    assert Path(BUNDLE).exists(), f"upload whospoke_colab_bundle.zip to {BUNDLE}"
    shutil.copy(BUNDLE, "/content/bundle.zip")
    zipfile.ZipFile("/content/bundle.zip").extractall("/content")
    import soundfile as sf                                  # FLAC -> WAV (lossless; same 16-bit samples)
    flacs = list((REPO / "Data").rglob("*.wav.flac"))
    for f in flacs:
        pcm, sr = sf.read(f, dtype="int16")
        sf.write(str(f)[:-5], pcm, sr, subtype="PCM_16")
        f.unlink()
    print(f"unpacked; {len(flacs)} audio files restored to WAV")
# Keep results/ and models/ on Drive, so nothing is lost if Colab disconnects.
for name in ("results", "models"):
    keep, local = Path(OUT, name), REPO / name
    keep.mkdir(parents=True, exist_ok=True)
    if local.exists() and not local.is_symlink():
        shutil.copytree(local, keep, dirs_exist_ok=True)     # first run: seed Drive with the bundle's files
        shutil.rmtree(local)
    if not local.exists():
        local.symlink_to(keep, target_is_directory=True)
os.chdir(REPO)
data = REPO / "Data"
os.environ["WHOSPOKE_DATA"] = str(data)
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"      # torch >= 2.6: allow the pyannote/Asteroid checkpoints
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
# onnxruntime-gpu (speech-to-text) needs CUDA 12 + cuDNN 9; Colab's torch uses CUDA 13, so add the CUDA 12 libraries
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nvidia-cuda-runtime-cu12", "nvidia-cublas-cu12",
                "nvidia-cudnn-cu12", "nvidia-cufft-cu12", "nvidia-curand-cu12", "nvidia-cuda-nvrtc-cu12"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "asteroid==0.7.0"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tabulate"], check=True)
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])
from huggingface_hub import auth_check
for repo, kind in (("pyannote/segmentation-3.0", "model"), ("ai4bharat/indic-conformer-600m-multilingual", "model"),
                   ("ai4bharat/IndicVoices", "dataset")):
    auth_check(repo, repo_type=kind)                         # fails here, early, if the terms are not accepted
    print("access ok:", repo)
print("data:", data, "| outputs kept in:", OUT)

In [ ]:
import torch, pandas as pd
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "Switch the runtime to a T4 GPU (Runtime -> Change runtime type)"
for split in ("dev", "test"):
    idx = pd.read_csv(data / "synth" / split / "index.csv")
    print(f"{split}: {len(idx)} conversations, {idx.duration_s.sum() / 60:.0f} min")
for f in ("train-00000-of-00082.parquet", "valid-00000-of-00001.parquet"):
    print(f, "found" if (data / "raw" / "indicvoices" / "hindi" / f).exists() else "MISSING (will be downloaded)")

## 0b · One-minute check before the long run

Loads every model on a few seconds of audio. It must end with **ALL OK**. If it prints a WARNING that speech-to-text is
on the CPU, everything still works but sections 3 and 5 take several times longer.

In [ ]:
!python scripts/check_environment.py

## 1 · Fine-tune (dev picks the checkpoint)

Step 0 is the released model: its dev score must be **6.43 dB** (the published dev number). If it is not, stop and
report it, because something differs from the original setup.

In [ ]:
!python scripts/finetune_convtasnet.py --steps {STEPS} --batch {BATCH} --extra-shards {EXTRA_SHARDS}

In [ ]:
log = pd.read_csv("results/finetune_convtasnet/log.csv")
display(log)
print(f"released model: {log.dev_si_sdri.iloc[0]:.2f} dB   best fine-tuned: {log.dev_si_sdri.max():.2f} dB "
      f"(step {int(log.step[log.dev_si_sdri.idxmax()])})")

## 2 · Dev: separation quality, released vs fine-tuned

Same script and regions as the published Stage-1 numbers (`--tag hindi` writes new files; the published
`separation_dev.csv` is not touched).

In [ ]:
!python scripts/eval_separation.py --split dev --models convtasnet convtasnet-hindi --tag hindi

## 3 · Dev: does it lower the transcript error?

The splice policy with each separator, on the true timeline and on our diarization. Both runs are made here, on
the same machine, so the comparison is fair.

In [ ]:
# "mixture" (no separation) is the same whatever the separator, so it is run once, with the released model
!python scripts/tune_separation_policy.py --split dev --separator convtasnet --policies mixture splice-0.5 --out separation_policy_dev_convtasnet.csv
!python scripts/tune_separation_policy.py --split dev --separator convtasnet-hindi --policies splice-0.5 --out separation_policy_dev_convtasnet-hindi.csv

## 4 · Decision (rule 3, fixed in advance)

In [ ]:
R = Path("results")
def cpwer(name):
    d = pd.read_csv(R / name)
    d = d[d.policy == "splice-0.5"]
    return {tl: g.errors.sum() / g.ref_words.sum() for tl, g in d.groupby("timeline")}

sep = pd.read_csv(R / "separation_dev_hindi.csv")
ovl = sep[sep.view == "overlap-region"].groupby("model").si_sdri.mean()
base, hindi = cpwer("separation_policy_dev_convtasnet.csv"), cpwer("separation_policy_dev_convtasnet-hindi.csv")
checks = {
    "dev overlap SI-SDRi higher": bool(ovl["convtasnet-hindi"] > ovl["convtasnet"]),
    "dev cpWER true timeline lower": bool(hindi["oracle"] < base["oracle"]),
    "dev cpWER our diarization not higher": bool(hindi["B-spectral"] <= base["B-spectral"]),
}
decision = {"dev_overlap_si_sdri": {k: round(float(v), 3) for k, v in ovl.items()},
            "dev_cpwer_splice": {"convtasnet": {k: round(v, 4) for k, v in base.items()},
                                 "convtasnet-hindi": {k: round(v, 4) for k, v in hindi.items()}},
            "checks": checks, "wins_on_dev": all(checks.values())}
(R / "finetune_convtasnet" / "dev_decision.json").write_text(json.dumps(decision, indent=1))
print(json.dumps(decision, indent=1))
print("\n==> ", "WINS on dev: run section 5 (test, once)" if decision["wins_on_dev"] else
      "does NOT win on dev: the test set is not run (rule 4). Skip to section 6.")

## 5 · Test, once

The dev decision above is already saved. The test set is run once either way, so every experiment is complete; it
does not change the decision. The released model is run again here too, so the comparison is on the same machine.
Its numbers should be close to the published ones (overlap SI-SDRi +9.62 dB; cpWER 26.1 % true timeline, 49.5 % our
diarization).

In [ ]:
print("dev decision (already saved):", "WINS" if decision["wins_on_dev"] else "does not win")
!python scripts/eval_separation.py --split test --models convtasnet convtasnet-hindi --tag hindi
!python scripts/tune_separation_policy.py --split test --separator convtasnet --policies mixture splice-0.5 --out separation_policy_test_convtasnet.csv
!python scripts/tune_separation_policy.py --split test --separator convtasnet-hindi --policies splice-0.5 --out separation_policy_test_convtasnet-hindi.csv

## 6 · Report (written to `results/finetune_convtasnet/REPORT.md`)

In [ ]:
cfg = json.loads((R / "finetune_convtasnet" / "config.json").read_text())
lines = ["# A3 · Conv-TasNet fine-tuned on Hindi mixtures: results", "",
         f"GPU: {cfg['gpu']} · training pool: {cfg['pool']['speakers']} speakers, {cfg['pool']['hours']} h "
         f"({', '.join(cfg['pool']['shards'])}) · {cfg['excluded_speakers']} speakers held out · "
         f"steps run: {int(log.step.max())} of {cfg['steps']} · batch {cfg['batch']} · lr {cfg['lr']}", "",
         "## Training (dev overlap SI-SDRi, dB)", "", log.to_markdown(index=False), ""]

def sep_table(split):
    f = R / f"separation_{split}_hindi.csv"
    if not f.exists():
        return ["(not run)", ""]
    d = pd.read_csv(f)
    t = d.groupby(["view", "model", "noise"]).si_sdri.mean().unstack("noise").round(2)
    t["all"] = d.groupby(["view", "model"]).si_sdri.mean().round(2)
    t["n"] = d.groupby(["view", "model"]).size()
    return [t.to_markdown(), ""]

def cp_table(split):
    rows = []
    for s in ("convtasnet", "convtasnet-hindi"):
        f = R / f"separation_policy_{split}_{s}.csv"
        if not f.exists():
            continue
        d = pd.read_csv(f)
        for (tl, pol), g in d.groupby(["timeline", "policy"]):
            rows.append({"separator": s, "timeline": tl, "policy": pol,
                         "cpWER": round(g.errors.sum() / g.ref_words.sum(), 4),
                         "heavy overlap": round(g[g.overlap_level == "ovl-high"].errors.sum()
                                                / g[g.overlap_level == "ovl-high"].ref_words.sum(), 4)})
    return [pd.DataFrame(rows).to_markdown(index=False) if rows else "(not run)", ""]

lines += ["## Dev: separation (SI-SDRi, dB)", ""] + sep_table("dev")
lines += ["## Dev: cpWER", ""] + cp_table("dev")
lines += ["## Decision", "", "```json", json.dumps(decision, indent=1), "```", ""]
lines += ["## Test (run once, after the dev decision was saved; "
          + ("it WON on dev)" if decision["wins_on_dev"] else "it did NOT win on dev, so these numbers are for completeness)"), ""] + sep_table("test") + cp_table("test")
(R / "finetune_convtasnet" / "REPORT.md").write_text("\n".join(lines))
print("\n".join(lines))
print("\nDone. Download the folder", OUT, "(results/ and models/) and bring it back.")